# 03 — Point-in-time feature engineering

## What is this notebook for?

The SLA dataset contains powerful information that appears only **after** the ticket progresses.

Our challenge is:

> **Make the signal richer without letting the current ticket see its own future.**

We build five layers:

1. ticket/clock context;
2. SLA geometry;
3. service pressure;
4. historical burden and risk;
5. initial-description signals.

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print("Project root:", project_root)

from src.config import data_dir
from src.data.load import load_source_tables, build_master_table
from src.features.base import add_base_features, add_description_signals
from src.features.pressure import add_pressure_features
from src.features.historical import add_outcome_teacher_columns, causal_historical_features

Project root: /mnt/data/northbridge-sla-breach-work/northbridge-sla-breach


## 1. Start from the same master table

In [2]:
tickets, agents, clients = load_source_tables(data_dir())
master = build_master_table(tickets, agents, clients)
master[["TicketID","CreatedAt","PriorityID","CategoryID","Channel","DailyCapacity","SLADueAt","SLABreached"]].head()

,TicketID,CreatedAt,PriorityID,CategoryID,Channel,DailyCapacity,SLADueAt,SLABreached
0,2179,2023-01-01 10:54:00,3,5,Form,27,2023-01-04 10:54:00,0
1,322,2023-01-01 11:11:00,3,2,Email,24,2023-01-03 11:11:00,0
2,1331,2023-01-01 20:04:00,2,4,Portal,20,2023-01-02 08:04:00,0
3,493,2023-01-01 21:11:00,3,1,Email,19,2023-01-04 21:11:00,0
4,2022,2023-01-04 04:49:00,3,1,Email,19,2023-01-07 04:49:00,0


## 2. Layer 1 — clock and urgency features

Clock time is circular: 23:00 and 00:00 are neighbours even though 23 and 0 look far apart numerically.

We also reverse priority into an intuitive urgency score:

```text
Critical=4, High=3, Standard=2, Low=1
```

In [3]:
base = add_base_features(master, sla_due_known=True)
base[["TicketID","CreatedAt","created_hour","is_weekend","is_after_hours",
      "PriorityID","priority_urgency","hour_sin","hour_cos"]].head()

,TicketID,CreatedAt,created_hour,is_weekend,is_after_hours,PriorityID,priority_urgency,hour_sin,hour_cos
0,2179,2023-01-01 10:54:00,10,1,1,3,2,0.500,-0.866
1,322,2023-01-01 11:11:00,11,1,1,3,2,0.259,-0.966
2,1331,2023-01-01 20:04:00,20,1,1,2,3,-0.866,0.500
3,493,2023-01-01 21:11:00,21,1,1,3,2,-0.707,0.707
4,2022,2023-01-04 04:49:00,4,0,1,3,2,0.866,0.500


## 3. Layer 2 — SLA geometry

Rather than only knowing the deadline timestamp, calculate how much time exists between creation and deadline.

In [4]:
base[["TicketID","CreatedAt","SLADueAt","SLAWindowHours",
      "due_hour","due_day_of_week","due_on_weekend"]].head()

,TicketID,CreatedAt,SLADueAt,SLAWindowHours,due_hour,due_day_of_week,due_on_weekend
0,2179,2023-01-01 10:54:00,2023-01-04 10:54:00,72.000,10,2,0
1,322,2023-01-01 11:11:00,2023-01-03 11:11:00,48.000,11,1,0
2,1331,2023-01-01 20:04:00,2023-01-02 08:04:00,12.000,8,0,0
3,493,2023-01-01 21:11:00,2023-01-04 21:11:00,72.000,21,2,0
4,2022,2023-01-04 04:49:00,2023-01-07 04:49:00,72.000,4,5,1


`SLAWindowHours` is closer to the physical breach mechanism:

> **How much time is this ticket actually allowed?**

## 4. Layer 3 — service pressure

DailyCapacity is supply. We also need demand.

At each ticket's creation time we reconstruct:

- earlier unresolved cases for its agent/team;
- arrivals during the previous 1 and 4 hours;
- backlog relative to capacity.

Future tickets never count toward the present ticket.

In [5]:
pressure = add_pressure_features(base, agents)
pressure[["TicketID","DailyCapacity","agent_open_backlog","team_open_backlog",
          "agent_arrivals_1h","team_arrivals_4h",
          "agent_backlog_per_capacity","team_backlog_per_capacity"]].iloc[995:1000].round(3)

,TicketID,DailyCapacity,agent_open_backlog,team_open_backlog,agent_arrivals_1h,team_arrivals_4h,agent_backlog_per_capacity,team_backlog_per_capacity
995,2126,18,1.000,9.000,0.000,0.000,0.056,0.019
996,3447,24,0.000,9.000,0.000,0.000,0.000,0.019
997,990,14,1.000,18.000,0.000,0.000,0.071,0.023
998,3378,19,0.000,19.000,0.000,1.000,0.000,0.024
999,1778,18,0.000,20.000,0.000,1.000,0.000,0.026


A capacity of 15 is not automatically high or low.

`agent_backlog_per_capacity` gives it operational context.

## 5. Outcome fields as historical teachers

`FirstResponseAt`, `ResolvedAt` and `ResolutionNotes` are illegal predictors for the **current** ticket.

But completed old tickets can teach us how difficult similar work historically was.

For old cases we derive:

- response delay;
- post-response resolution span;
- simple note themes such as investigation, approval, financial work, documentation and escalation.

In [6]:
teacher = add_outcome_teacher_columns(master)
teacher[["TicketID","CategoryID","Channel","response_hours","resolution_hours",
         "note_action_count","ResolutionNotes"]].dropna(subset=["resolution_hours"]).head(5)

,TicketID,CategoryID,Channel,response_hours,resolution_hours,note_action_count,ResolutionNotes
0,2179,5,Form,4.800,35.850,3,Clinical summary prepared and sent to patient ...
1,322,2,Email,3.083,24.767,3,Invoice reviewed and corrected. Credit note is...
2,1331,4,Portal,1.617,5.900,3,Complaint resolved following discussion with c...
3,493,1,Email,6.817,58.283,0,Recurring appointment series updated following...
4,2022,1,Email,5.750,36.483,1,Urgent appointment arranged within 24 hours fo...


These are **teacher variables**, not direct T1 predictors.

The current ticket receives only summaries learned from cases that had already finished before it arrived.

## 6. Layer 4 — causal historical burden and target encoding

For each ticket we ask:

> What had already-resolved similar tickets taught us **before this ticket was created**?

Examples:

- median historical resolution burden for `Category × Channel`;
- historical breach rate for `Category × Channel`;
- historical breach rate for `Agent × Category`;
- support counts showing how much history exists.

The breach rates are smoothed toward the historical global rate to reduce instability in tiny groups.

In [7]:
history = causal_historical_features(pressure)
feature_step = pd.concat([pressure, history], axis=1)
feature_step[["TicketID","CategoryID","Channel","AssignedAgentID",
              "hist_resolution_burden","hist_burden_support",
              "hist_cc_breach_rate","hist_cc_breach_rate_support",
              "hist_agent_category_breach_rate",
              "hist_agent_category_breach_rate_support"]].iloc[1000:1005].round(3)

,TicketID,CategoryID,Channel,AssignedAgentID,hist_resolution_burden,hist_burden_support,hist_cc_breach_rate,hist_cc_breach_rate_support,hist_agent_category_breach_rate,hist_agent_category_breach_rate_support
1000,318,3,Form,53,23.167,24.000,0.180,24.000,0.178,2.000
1001,3154,4,Phone,70,18.000,26.000,0.215,26.000,0.163,4.000
1002,77,1,Email,102,33.775,110.000,0.176,110.000,0.219,7.000
1003,2978,4,Email,113,15.817,32.000,0.248,32.000,0.187,1.000
1004,1904,2,Portal,73,27.892,70.000,0.188,70.000,0.220,7.000


### Why this is leakage-safe target encoding

Unsafe:

```text
whole dataset outcomes -> group mean -> every row
```

Safe:

```text
past completed outcomes -> historical estimate -> current row
```

The current row never contributes its own target to its encoded value.

## 7. CategoryID remains nominal

Category 5 is not mathematically “greater” than Category 4.

So we keep CategoryID as a label and create a separate quantity:

```text
hist_resolution_burden
```

That has a meaningful unit: the historical resolution span of similar already-completed work.

## 8. Layer 5 — initial description signal

The initial Description exists at creation, so it is legal T1 information.

We start with simple explainable text signals rather than immediately using a complex language model.

In [8]:
signal = add_description_signals(feature_step)
signal[["TicketID","CategoryID","Description","description_word_count",
        "description_has_currency","description_has_insurer",
        "description_has_document","description_has_appointment"]].head(5)

,TicketID,CategoryID,Description,description_word_count,description_has_currency,description_has_insurer,description_has_document,description_has_appointment
0,2179,5,Request for discharge summary for patient PT-4...,7.000,0,0,1,0
1,322,2,Query regarding invoice 51017 dated 15 Dec 202...,14.000,1,0,1,0
2,1331,4,Formal complaint submitted by patient regardin...,11.000,0,0,0,0
3,493,1,Patient PT-17762 requesting cancellation of ap...,12.000,0,0,0,1
4,2022,1,Patient PT-60009 requesting cancellation of ap...,12.000,0,0,0,1


## 9. Mechanism-based interactions

We create a few interactions that have clear business meaning:

- Category × Channel;
- Priority × Contract;
- historical burden ÷ SLA window.

The last one asks whether similar work historically consumes a large fraction of the time allowed.

In [9]:
features = signal.copy()
features["category_channel"] = features["CategoryID"].astype(str)+"__"+features["Channel"].astype(str)
features["priority_contract"] = features["PriorityID"].astype(str)+"__"+features["ContractTier"].astype(str)
features["burden_to_sla"] = features["hist_resolution_burden"] / features["SLAWindowHours"].replace(0,np.nan)
features[["TicketID","category_channel","priority_contract",
          "hist_resolution_burden","SLAWindowHours","burden_to_sla"]].iloc[1000:1005].round(3)

,TicketID,category_channel,priority_contract,hist_resolution_burden,SLAWindowHours,burden_to_sla
1000,318,3__Form,1__Premium,23.167,4.000,5.792
1001,3154,4__Phone,3__Standard,18.000,72.000,0.250
1002,77,1__Email,3__Standard,33.775,72.000,0.469
1003,2978,4__Email,2__Premium,15.817,12.000,1.318
1004,1904,2__Portal,1__Premium,27.892,4.000,6.973


## 10. What one engineered row now tells the model

In [10]:
features[["TicketID","PriorityID","CategoryID","Channel","SLAWindowHours",
          "agent_open_backlog","agent_backlog_per_capacity",
          "hist_resolution_burden","hist_cc_breach_rate",
          "hist_agent_category_breach_rate","description_word_count",
          "SLABreached"]].iloc[1000:1005].round(3)

,TicketID,PriorityID,CategoryID,Channel,SLAWindowHours,agent_open_backlog,agent_backlog_per_capacity,hist_resolution_burden,hist_cc_breach_rate,hist_agent_category_breach_rate,description_word_count,SLABreached
1000,318,1,3,Form,4.000,0.000,0.000,23.167,0.180,0.178,9.000,0
1001,3154,3,4,Phone,72.000,1.000,0.043,18.000,0.215,0.163,11.000,0
1002,77,3,1,Email,72.000,1.000,0.056,33.775,0.176,0.219,12.000,0
1003,2978,2,4,Email,12.000,0.000,0.000,15.817,0.248,0.187,11.000,1
1004,1904,1,2,Portal,4.000,0.000,0.000,27.892,0.188,0.220,14.000,0


## 11. Feature-engineering checkpoint

We have moved from:

```text
Who/what is this ticket?
```

to:

```text
What work is this?
How much time is available?
How busy is the service operation?
How difficult were similar completed cases?
How risky has this combination been historically?
```

But a plausible feature is still only a **hypothesis**.

It must be:

1. available at prediction time;
2. business-defensible;
3. useful out of time.

**Next:** Notebook 04 tests each signal layer rather than assuming it helps.